In [8]:
import pandas as pd

cps = pd.read_csv('CLEANED_CPS_2005_2023.csv')
fmr = pd.read_csv('FMR_clean.csv')
hpi = pd.read_csv('cleaned_hpi_2005-2023.csv')
acs = pd.read_csv('ACS_FLATTENED.csv')


cbsa_map = {
    'Bergen': [35620], 'Essex': [35620], 'Hudson': [35620], 'Hunterdon': [35620],      #|
    'Middlesex': [35620], 'Monmouth': [35620], 'Morris': [35620], 'Ocean': [35620],    #| Section 1
    'Passaic': [35620], 'Somerset': [35620], 'Sussex': [35620], 'Union': [35620],      #|
    'Burlington': [37980], 'Camden': [37980], 'Gloucester': [37980], 'Salem': [37980], #| Section 2
    'Mercer': [45940],                                                                 #| Section 3
    'Atlantic': [12100],                                                               #| Section 4
    'Cumberland': [47220],                                                             #| Section 5
    'Warren': [10900],                                                                 #| Section 6
    'Cape May': [0, 36140]                                                             #| Section 7
}

fmr['CBSA'] = fmr['county'].map(cbsa_map)
fmr = fmr.explode('CBSA') 

fmr_full = fmr.groupby(['year', 'CBSA'])[[
    
    'fmr_0br','fmr_1br','fmr_2br',
    'fmr_3br','fmr_4br','fmr_percentile'

]].mean().reset_index()

hpi['CBSA'] = hpi['County'].map(cbsa_map)
hpi = hpi.explode('CBSA')

hpi_full = hpi.groupby(['Year', 'CBSA'])[[

    'Annual Change (%)','HPI',
    'HPI with 1990 base','HPI with 2000 base'

    ]].mean().reset_index()

acs['CBSA'] = acs['county_clean'].map(cbsa_map)
acs = acs.explode('CBSA')

acs_full = acs.groupby(['year', 'CBSA'])[[

    'total_occupied_units', 'owner_occupied_total', 'renter_occupied_total','owner_occupied_hh_15_24', 'owner_occupied_hh_25_34', 'owner_occupied_hh_35_44','owner_occupied_hh_45_54', 
    'owner_occupied_hh_55_59', 'owner_occupied_hh_60_64','owner_occupied_hh_65_74', 'owner_occupied_hh_75_84', 'owner_occupied_hh_85_plus','renter_occupied_hh_15_24', 'renter_occupied_hh_25_34', 
    'renter_occupied_hh_35_44','renter_occupied_hh_45_54', 'renter_occupied_hh_55_59', 'renter_occupied_hh_60_64','renter_occupied_hh_65_74', 'renter_occupied_hh_75_84', 'renter_occupied_hh_85_plus'

]].sum().reset_index()

acs_full['owner_rate'] = acs_full['owner_occupied_total'] / acs_full['total_occupied_units']
acs_full['renter_rate'] = acs_full['renter_occupied_total'] / acs_full['total_occupied_units']

fmr_full = fmr_full.rename(columns={'year': 'Year'})
hpi_full = hpi_full.rename(columns={'year': 'Year'})
acs_full = acs_full.rename(columns={'year': 'Year'})


fmr_full.to_csv('FMR_CBSA_GROUPED.csv', index=False)
hpi_full.to_csv('HPI_CBSA_GROUPED.csv', index=False)
acs_full.to_csv('ACS_CBSA_GROUPED.csv', index=False)


In [11]:
import pandas as pd

cps = pd.read_csv('CLEANED_CPS_2005_2023.csv')
fmr = pd.read_csv('FMR_CBSA_GROUPED.csv')
hpi = pd.read_csv('HPI_CBSA_GROUPED.csv')
acs = pd.read_csv('ACS_CBSA_GROUPED.csv')
output_filename = 'FINAL_MERGED_DATASET.csv'

cps = cps.rename(columns={'County': 'CBSA'})

cps_merged = pd.merge(cps, fmr, on=['Year', 'CBSA'], how='left')

cps_merged = pd.merge(cps_merged, hpi, on=['Year', 'CBSA'], how='left')

cps_merged = pd.merge(cps_merged, acs, on=['Year', 'CBSA'], how='left')

cps_merged.to_csv(output_filename, index=False)


In [ ]:
import pandas as pd

df = pd.read_csv('FINAL_MERGED_DATASET.csv')

# 2. 2015 Baseline Median Prices (ACS B25077)
anchor_prices_2015 = {
    35620: 414000,  # NYC-Newark-Jersey City 
    37980: 240900,  # Phila-Camden           
    45940: 283600,  # Trenton                
    12100: 223400,  # Atlantic City          
    47220: 157300,  # Vineland               
    10900: 203500,  # Allentown              
    36140: 311800,  # Ocean City  
    0: 311800       # Ocean City    
}

mortgage_rates = {
    2005: 5.87, 2006: 6.41, 2007: 6.34, 2008: 6.03,
    2009: 5.04, 2010: 4.69, 2011: 4.45, 2012: 3.66,
    2013: 3.98, 2014: 4.17, 2015: 3.85, 2016: 3.65, 
    2017: 3.99, 2018: 4.54, 2019: 3.94, 2020: 3.11,
    2021: 2.96, 2022: 5.34, 2023: 6.81
}

hpi_baseline = df[df['Year'] == 2015].drop_duplicates('CBSA').set_index('CBSA')['HPI'].to_dict()

def calculate_real_price(row):

    cbsa = row['CBSA']
    current_hpi = row['HPI']
 
    if pd.isna(current_hpi) or cbsa not in hpi_baseline or cbsa not in anchor_prices_2015:
        
        return None
        
    anchor_hpi = hpi_baseline[cbsa]
    anchor_price = anchor_prices_2015[cbsa]

    return (current_hpi / anchor_hpi) * anchor_price

df['Estimated_Median_Home_Price'] = df.apply(calculate_real_price, axis=1)

df['Mortgage_Interest_Rate'] = df['Year'].map(mortgage_rates)

df.to_csv('FINAL_MERGED_DATASE_HH_MEDIANS.csv', index=False)


In [4]:

import pandas as pd

df = pd.read_csv('FINAL_MERGED_DATASET_HH_MEDIANS.csv')

annual_cpi = {
    2005: 195.3, 2006: 201.6, 2007: 207.3, 2008: 215.3, 2009: 214.5,
    2010: 218.1, 2011: 224.9, 2012: 229.6, 2013: 233.0, 2014: 236.7,
    2015: 237.0, 2016: 240.0, 2017: 245.1, 2018: 251.1, 2019: 255.7,
    2020: 258.8, 2021: 271.0, 2022: 292.7, 2023: 304.7
}

cols_to_adjust = [
    'HH_Income', 
    'Person_Income', 
    'fmr_0br', 
    'fmr_1br', 
    'fmr_2br', 
    'fmr_3br', 
    'fmr_4br', 
    'Estimated_Median_Home_Price'
]

recent_cpi = annual_cpi[2023]

def calculate_inflation_multiplier(year):

    if pd.isna(year) or year not in annual_cpi:

        return None
    
    current_cpi = annual_cpi[year]
    
    return recent_cpi / current_cpi

df['Inflation_Rate'] = df['Year'].apply(calculate_inflation_multiplier)

for col in cols_to_adjust:

    if col in df.columns:

        new_col_name = f"Adjusted_{col}"
        df[new_col_name] = df[col] * df['Inflation_Rate']
        df[new_col_name] = df[new_col_name].round(2)

output_name = 'FINAL_INFLATED_MERGED_DATASET.csv'
df.to_csv(output_name, index=False)



In [6]:
tenure_codes = df["Tenure"].dropna().unique()

print("Unique tenure codes:")

for tenure in sorted(tenure_codes):
    print(tenure)

Unique tenure codes:
1
2
3


1. own
2. rent cash
3. rent not cash

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('FINAL_INFLATED_MERGED_DATASET_step3.csv')

def calculate_monthly_mortgage(row):

    home_price = row.get('Adjusted_Estimated_Median_Home_Price', np.nan)
    annual_rate = row.get('Mortgage_Interest_Rate', np.nan)
    
    if pd.isna(home_price) or pd.isna(annual_rate):

        return np.nan
    
    principal = home_price * 0.80 # 20% down 
    r = (annual_rate / 100) / 12
    n = 360
    
    if r > 0:

        return principal * (r * (1 + r)**n) / ((1 + r)**n - 1)
    
    return principal / n 

df['Monthly_Mortgage'] = df.apply(calculate_monthly_mortgage, axis=1).round(2)

def get_monthly_rent(row):

    hh_size = row.get('HH_Size', 2) 
  
    if hh_size == 1:

        return row.get('Adjusted_fmr_1br', np.nan)
    
    elif hh_size == 2:

        return row.get('Adjusted_fmr_2br', np.nan)
    
    elif hh_size in [3, 4]:

        return row.get('Adjusted_fmr_3br', np.nan)
    
    elif hh_size >= 5:

        return row.get('Adjusted_fmr_4br', np.nan)
    
    return np.nan

df['Monthly_Rent'] = df.apply(get_monthly_rent, axis=1).round(2)

def get_housing_cost(row):

    tenure = str(row.get('Tenure', '')).strip()
    
    if tenure.startswith('1'):

        return row['Monthly_Mortgage']
        
    elif tenure.startswith('2'):

        return row['Monthly_Rent']
        
    elif tenure.startswith('3'):

        return 0.0
        
    return np.nan

df['Real_Monthly_Housing_Cost_2023'] = df.apply(get_housing_cost, axis=1)
df.to_csv('FINAL_EDA_READY_DATASET.csv', index=False)


In [ ]:
import pandas as pd
import numpy as np

# 1. Load the dataset that now contains the monthly costs
df = pd.read_csv('FINAL_EDA_READY_DATASET.csv')

# 2. Function to Calculate Affordability/Burden
def calculate_affordability(row):
    income = row.get('Real_HH_Income_2023', np.nan)
    monthly_cost = row.get('Real_Monthly_Housing_Cost_2023', np.nan)
    
    # If income or cost is missing (or if income is 0), we can't do the math
    if pd.isna(income) or income <= 0 or pd.isna(monthly_cost):
        return pd.Series({'Housing_Cost_Ratio': np.nan, 'Is_Cost_Burdened': np.nan})
    
    # Multiply the monthly cost by 12 to compare it against Annual Income
    annual_housing_cost = monthly_cost * 12
    ratio = annual_housing_cost / income
    
    # Flag 1 if they spend 30% or more of their income on housing, 0 if under
    is_burdened = 1 if ratio >= 0.30 else 0
    
    return pd.Series({'Housing_Cost_Ratio': round(ratio, 4), 'Is_Cost_Burdened': is_burdened})

new_cols = df.apply(calculate_affordability, axis=1)
df = pd.concat([df, new_cols], axis=1)

# 3. Save the final dataset
df.to_csv('FINAL_EDA_READY_DATASET.csv', index=False)
print("✅ Step 2 Complete: Affordability ratio and burden flags added!")